# Microsoft 365 Usage — Gold (template inputs)

Builds the five files the **M365 Usage Analytics** Power BI template reads, from the tables `M365_Usage_Audit_Ingester` writes:

| Template parameter | Delta table | CSV in `Files/m365_usage_export/` |
|---|---|---|
| Rollup | `dbo.m365_usage_rollup` | `Rollup.csv` |
| Session Stats | `dbo.m365_usage_sessionstats` | `SessionStats.csv` |
| User Stats | `dbo.m365_usage_userstats` | `UserStats.csv` |
| Session Cohort | `dbo.m365_usage_sessioncohort` | `SessionCohort.csv` |
| Entra Users Data | `dbo.m365_usage_entrausers` | `EntraUsers.csv` |

*User Stats* and *Session Cohort* come from the processor's own `write_userstats_files`, embedded unchanged. *Entra Users* is read from Microsoft Graph
with the columns the template expects; the custom HR columns are left blank.

**Permissions**: `User.Read.All` (application, admin-consented).

## 1. Configuration

In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'

ROLLUP_TABLE    = 'dbo.m365_usage_rollup'
SESSION_TABLE   = 'dbo.m365_usage_sessionstats'
USERSTATS_TABLE = 'dbo.m365_usage_userstats'
COHORT_TABLE    = 'dbo.m365_usage_sessioncohort'
ENTRA_TABLE     = 'dbo.m365_usage_entrausers'
EXPORT_DIR      = 'Files/m365_usage_export'   # CSVs for the template's file parameters

## 2. Authenticate to Microsoft Graph

In [ ]:
import requests, time


def get_graph_token(tenant_id, client_id, client_secret):
    url = f'https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token'
    body = {
        'client_id': client_id,
        'scope': 'https://graph.microsoft.com/.default',
        'client_secret': client_secret,
        'grant_type': 'client_credentials',
    }
    r = requests.post(url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']


def _auth_headers():
    return {'Authorization': 'Bearer ' + token}


token = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
print('✓ Graph token acquired.')

GRAPH_MAX_ATTEMPTS = 6
GRAPH_MAX_WAIT_SEC = 120


def graph_get(url, timeout=60):
    """GET from Graph; waits out throttling and brief outages and renews an expired token once."""
    global token
    renewed = False
    for attempt in range(1, GRAPH_MAX_ATTEMPTS + 1):
        last = attempt == GRAPH_MAX_ATTEMPTS
        try:
            r = requests.get(url, headers=_auth_headers(), timeout=timeout)
        except (requests.ConnectionError, requests.Timeout) as exc:
            if last:
                raise
            wait = min(5 * 2 ** attempt, GRAPH_MAX_WAIT_SEC)
            print(f'  Graph request failed ({type(exc).__name__}); trying again in {wait}s.')
            time.sleep(wait)
            continue
        if r.status_code == 401 and not renewed and not last:
            token = get_graph_token(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
            renewed = True
            continue
        if (r.status_code == 429 or r.status_code >= 500) and not last:
            try:
                wait = float(r.headers.get('Retry-After', ''))
            except (TypeError, ValueError):
                wait = 5 * 2 ** attempt
            wait = min(max(wait, 1.0), GRAPH_MAX_WAIT_SEC)
            print(f'  Graph returned {r.status_code}; trying again in {wait:.0f}s.')
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r

## 3. Entra users

Pages through `/users` with the manager expanded and shapes each user into the template's *Entra Users* columns.

In [ ]:
import json

ENTRA_COLUMNS = [
    'userPrincipalName', 'displayName', 'id', 'mail', 'givenName', 'surname', 'jobTitle', 'department',
    'employeeType', 'employeeId', 'employeeHireDate', 'officeLocation', 'city', 'state', 'country',
    'postalCode', 'companyName', 'employeeOrgData_division', 'employeeOrgData_costCenter', 'accountEnabled',
    'userType', 'createdDateTime', 'usageLocation', 'preferredLanguage', 'onPremisesSyncEnabled',
    'onPremisesImmutableId', 'externalUserState', 'proxyAddresses_Primary', 'proxyAddresses_Count',
    'proxyAddresses_All', 'manager_id', 'manager_displayName', 'manager_userPrincipalName', 'manager_mail',
    'manager_jobTitle', 'assignedLicenses', 'hasLicense',
    # Custom HR columns the template can show; not in Entra, so left blank.
    'ManagerID', 'BusinessAreaLabel', 'CountryofEmployment', 'CompanyCodeLabel', 'CostCentreLabel',
    'UserName', 'EffectiveDate', 'FunctionType', 'BusinessAreaCode', 'OrgLevel_3Label',
]
_select = ','.join([
    'userPrincipalName', 'displayName', 'id', 'mail', 'givenName', 'surname', 'jobTitle', 'department',
    'employeeType', 'employeeId', 'employeeHireDate', 'officeLocation', 'city', 'state', 'country',
    'postalCode', 'companyName', 'employeeOrgData', 'accountEnabled', 'userType', 'createdDateTime',
    'usageLocation', 'preferredLanguage', 'onPremisesSyncEnabled', 'onPremisesImmutableId',
    'externalUserState', 'proxyAddresses', 'assignedLicenses',
])
next_url = (f'https://graph.microsoft.com/v1.0/users?$select={_select}'
            '&$expand=manager($select=id,displayName,userPrincipalName,mail,jobTitle)&$top=999')

users, seen_urls = [], set()
while next_url:
    if next_url in seen_urls:
        raise ValueError('Graph /users paging loop detected.')
    seen_urls.add(next_url)
    data = graph_get(next_url).json()
    if not isinstance(data, dict) or not isinstance(data.get('value'), list):
        raise ValueError('Graph /users returned an unexpected page; refusing to write a partial user list.')
    users.extend(data['value'])
    next_url = data.get('@odata.nextLink')
    print(f'  Fetched {len(users):,} users so far...')
print(f'✓ Total users: {len(users):,}')


def _s(value):
    if value is None:
        return ''
    if isinstance(value, bool):
        return 'True' if value else 'False'
    return str(value)


def entra_row(u):
    org = u.get('employeeOrgData') or {}
    mgr = u.get('manager') or {}
    proxies = [p for p in (u.get('proxyAddresses') or []) if isinstance(p, str)]
    primary = next((p[5:] for p in proxies if p.startswith('SMTP:')), '')
    licenses = [{'skuId': l.get('skuId')} for l in (u.get('assignedLicenses') or []) if isinstance(l, dict)]
    row = {k: _s(u.get(k)) for k in ENTRA_COLUMNS if k in u and not isinstance(u.get(k), (dict, list))}
    row.update({
        'employeeOrgData_division': _s(org.get('division')),
        'employeeOrgData_costCenter': _s(org.get('costCenter')),
        'proxyAddresses_Primary': primary,
        'proxyAddresses_Count': str(len(proxies)),
        'proxyAddresses_All': ';'.join(proxies),
        'manager_id': _s(mgr.get('id')),
        'manager_displayName': _s(mgr.get('displayName')),
        'manager_userPrincipalName': _s(mgr.get('userPrincipalName')),
        'manager_mail': _s(mgr.get('mail')),
        'manager_jobTitle': _s(mgr.get('jobTitle')),
        'assignedLicenses': json.dumps(licenses),
        # The template recomputes hasLicense from assignedLicenses against its Copilot SKU list.
        'hasLicense': 'True' if licenses else 'False',
    })
    return [row.get(k, '') for k in ENTRA_COLUMNS]


entra_rows = [entra_row(u) for u in users if u.get('userPrincipalName')]

## 4. Microsoft 365 usage statistics (embedded processor)

The processor's own `write_userstats_files`, included verbatim (MIT licence).

In [ ]:
# Microsoft 365 usage row logic - copied verbatim from the M365 Usage Analytics processor.
# Source: https://github.com/microsoft/M365UsageAnalytics
#         scripts/Purview_M365_Usage_Bundle_Explosion_Processor_v2.6.1.py
# Copyright (c) Microsoft Corporation. Licensed under the MIT License.
# Only the functions and constants these notebooks call are included; they are unchanged so the
# numbers match the template's own pipeline. Regenerate with scripts/gen_m365_usage_notebooks.py.
from __future__ import annotations

import bisect
import csv
import os
import re
import sys
import time
from collections import defaultdict
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from typing import Any


EXPLOSION_PER_RECORD_ROW_CAP = 1000


ROLLUP_HEADER: list[str] = [
    "UserId", "CreationDate", "Operation", "Workload",
    "SourceFileExtension", "AppHost",
    "EventCount", "ItemsAccessedCount", "CreationTime", "MaxCreationTime",
    "AgentId", "AgentName", "ContextType", "IsAgentInteraction",
]


OP_RENAME: dict[str, str] = {
    "FileViewed":                "FileAccessed",
    "MeetingParticipantJoined":  "MeetingParticipantDetail",
    "ConnectedAIAppInteraction": "AIAppInteraction",
}


WORD_EXTS: set[str] = {"docx", "doc", "dotx"}


EXCEL_EXTS: set[str] = {"xlsx", "xls", "xlsm", "csv"}


PPT_EXTS: set[str] = {"pptx", "ppt", "ppsx"}


OFFICE_EXTS: set[str] = WORD_EXTS | EXCEL_EXTS | PPT_EXTS


FILE_OPS: set[str] = {
    "FileAccessed",                            # canonical (was FileViewed in legacy)
    "FileModified",
    "FileDownloaded",
    "FileUploaded",
}


OUTLOOK_OPS: set[str] = {"Send", "MailItemsAccessed", "MailboxLogin"}


TEAMS_OPS: set[str] = {
    "MessageSent",                             # Msgs Sent
    "MessageRead", "ChatCreated",              # Msgs Read (Graph-API tenants emit ChatCreated)
    "MeetingParticipantDetail",                # canonical (was MeetingParticipantJoined)
    "TeamsSessionStarted",                     # Meetings/calls fallback
}


COPILOT_OPS: set[str] = {"CopilotInteraction", "AIAppInteraction"}


AGENT_APPHOSTS: set[str] = {"agent", "copilotstudio", "declarativeagent", "customengineagent"}


DAX_FILE_OPS: set[str] = {
    "FileAccessed",      # canonical of legacy FileViewed (DAX checks both)
    "FilePreviewed",
    "FileModified",
    "FileDownloaded",
    "FileUploaded",
}


DAX_OUTLOOK_OPS: set[str] = {"Send", "MailItemsAccessed"}


DAX_TEAMS_OPS: set[str] = {
    "MessageSent", "MessageRead", "MessagesListed", "ChatRetrieved",
    "MeetingParticipantDetail",  # canonical of MeetingParticipantJoined
    "MeetingStarted", "MeetingEnded", "TeamsSessionStarted",
}


USERSTATS_HEADER: list[str] = [
    "UserId",
    "CopilotEC", "M365EC", "ExCopEC", "ExM365EC",
    "IsCopilotUser", "CopilotTierColumn", "M365TierColumn",
    "PriorityScatterColumn", "ExcelPriority",
    "CopilotUsageRankColumn", "M365UsageRankColumn",
    "TeamsActiveDays", "OutlookActiveDays", "WordActiveDays",
    "ExcelActiveDays", "PowerPointActiveDays",
    "TeamsActivityCount", "OutlookActivityCount", "OfficeFilesActivityCount",
    "TeamsActivitySegment", "OutlookActivitySegment", "WordActivitySegment",
    "ExcelActivitySegment", "PowerPointActivitySegment",
    "OfficeFilesActivitySegment", "OverallM365ActivitySegment",
    # ── v2.5.0: precomputed raw activity counts + CE percentile ranks per window.
    # Windows: _L30 = trailing 30 days ending at max(CreationDate); _L60 = trailing 60;
    # _Full  = entire data range. Filters match the corresponding DAX measures exactly
    # (post-canonicalization). CE ranks are integer 0-100; blank when raw is 0.
    "TeamsRaw_L30", "TeamsRaw_L60", "TeamsRaw_Full",
    "OutlookRaw_L30", "OutlookRaw_L60", "OutlookRaw_Full",
    "WordRaw_L30", "WordRaw_L60", "WordRaw_Full",
    "ExcelRaw_L30", "ExcelRaw_L60", "ExcelRaw_Full",
    "PowerPointRaw_L30", "PowerPointRaw_L60", "PowerPointRaw_Full",
    "CopilotChatRaw_L30", "CopilotChatRaw_L60", "CopilotChatRaw_Full",
    "CERank_Teams_L30", "CERank_Teams_L60", "CERank_Teams_Full",
    "CERank_Outlook_L30", "CERank_Outlook_L60", "CERank_Outlook_Full",
    "CERank_Word_L30", "CERank_Word_L60", "CERank_Word_Full",
    "CERank_Excel_L30", "CERank_Excel_L60", "CERank_Excel_Full",
    "CERank_PowerPoint_L30", "CERank_PowerPoint_L60", "CERank_PowerPoint_Full",
    "CERank_M365AllApps_L30", "CERank_M365AllApps_L60", "CERank_M365AllApps_Full",
    "CECopilotPercentile_L30", "CECopilotPercentile_L60", "CECopilotPercentile_Full",
]


RANK_WINDOWS: tuple[str, ...] = ("L30", "L60", "Full")


SESSIONCOHORT_HEADER: list[str] = ["UserId", "AppColumn", "SessionCohort"]


SESSIONSTATS_HEADER: list[str] = [
    "UserId", "CreationDate", "AppHost",
    "SessionCount", "PromptCount", "AgentPromptCount",
    "ResponseCount", "AgentSessionCount",
]


_CREATION_DATE_FORMATS: tuple[str, ...] = (
    "%Y-%m-%dT%H:%M:%S.%fZ",
    "%Y-%m-%dT%H:%M:%SZ",
    "%Y-%m-%dT%H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%S",
    "%m/%d/%Y %I:%M:%S %p",
    "%m/%d/%Y %H:%M:%S",
    "%Y-%m-%d",
    "%m/%d/%Y",
)


GroupKey = tuple[str, str, str, str, str, str, str, str, str]


def normalize_creation_date(raw: str) -> str:
    """Parse any Purview date format → 'YYYY-MM-DDT00:00:00.000Z' (midnight UTC)."""
    if not raw or not isinstance(raw, str):
        return ""
    raw = raw.strip()
    if not raw:
        return ""
    for fmt in _CREATION_DATE_FORMATS:
        try:
            dt = datetime.strptime(raw, fmt)
            return dt.strftime("%Y-%m-%d") + "T00:00:00.000Z"
        except ValueError:
            continue
    # Fallback: try extracting date portion from ISO-like string
    if len(raw) >= 10 and raw[4:5] == "-":
        return raw[:10] + "T00:00:00.000Z"
    return raw


def _norm_key_str(val: Any) -> str:
    """Normalize a string value for use as a rollup key: strip whitespace, empty if None."""
    if val is None:
        return ""
    if not isinstance(val, str):
        val = str(val)
    val = val.strip()
    if val.lower() in ("null", "none"):
        return ""
    return val


def safe_get(obj: Any, key: str) -> Any:
    """Safely retrieve a property from a dict-like object."""
    if obj is None:
        return None
    if isinstance(obj, dict):
        return obj.get(key)
    return getattr(obj, key, None)


def format_date_purview(val: Any) -> str:
    """Format a date value to ISO 8601 UTC string."""
    if val is None:
        return ""
    if isinstance(val, str):
        val = val.strip()
        if not val:
            return ""
        # Try common Purview date formats
        for fmt in (
            "%Y-%m-%dT%H:%M:%S.%fZ",
            "%Y-%m-%dT%H:%M:%SZ",
            "%Y-%m-%dT%H:%M:%S.%f",
            "%Y-%m-%dT%H:%M:%S",
            "%m/%d/%Y %I:%M:%S %p",
            "%m/%d/%Y %H:%M:%S",
        ):
            try:
                dt = datetime.strptime(val, fmt)
                return dt.replace(tzinfo=timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
            except ValueError:
                continue
        return val  # Return as-is if no format matches
    return str(val)


def get_array_fast(obj: Any, key: str) -> list:
    """Extract an array property, always returning a list."""
    if obj is None:
        return []
    val = safe_get(obj, key)
    if val is None:
        return []
    if isinstance(val, list):
        return val
    if isinstance(val, (str, int, float, bool)):
        return []
    try:
        return list(val)
    except (TypeError, ValueError):
        return []


def is_copilot(op: str, wl: str) -> bool:
    """True if the row represents a Copilot or agent / connected-app event."""
    return wl == "Copilot" or op in COPILOT_OPS


def is_excel_file_op(ext: str, op: str) -> bool:
    """True if the row is a file operation on an Excel-family extension."""
    return (ext or "").lower() in EXCEL_EXTS and op in FILE_OPS


def app_column(ext: str, op: str, wl: str) -> str:
    """Classify a row into an application column for session cohort grouping."""
    e = (ext or "").lower()
    if e in WORD_EXTS and op in FILE_OPS:
        return "Word"
    if e in EXCEL_EXTS and op in FILE_OPS:
        return "Excel"
    if e in PPT_EXTS and op in FILE_OPS:
        return "PowerPoint"
    if wl == "Exchange" and op in OUTLOOK_OPS:
        return "Outlook"
    if wl == "MicrosoftTeams" and op in TEAMS_OPS:
        return "Teams"
    if wl == "Copilot" or op == "CopilotInteraction":
        return "Copilot"
    return "M365 All Apps"


def percentile_inc(values: list[float], p: float) -> float:
    """Inclusive linear interpolation — matches PQ List.Percentile and numpy 'linear'."""
    if not values:
        return 0.0
    sorted_vals = sorted(values)
    n = len(sorted_vals)
    idx = p * (n - 1)
    lo, hi = int(idx), min(int(idx) + 1, n - 1)
    return sorted_vals[lo] + (idx - lo) * (sorted_vals[hi] - sorted_vals[lo])


def tier_fn(cnt: float, p90: float, p75: float, p50: float, zero_is_bottom: bool) -> str:
    """Assign a percentile-tier label."""
    if zero_is_bottom and cnt == 0:
        return "Bottom 50%"
    if cnt >= p90:
        return "Top 10%"
    if cnt >= p75:
        return "10-25%"
    if cnt >= p50:
        return "25-50%"
    return "Bottom 50%"


def priority_fn(m365_tier: str, cop_tier: str) -> str:
    """Map (M365 tier, Copilot tier) pair to a priority label."""
    top2 = {"Top 10%", "10-25%"}
    if m365_tier in top2 and cop_tier in top2:
        return "Promoter"
    if m365_tier == "Top 10%" and cop_tier == "25-50%":
        return "High"
    if m365_tier == "Top 10%" and cop_tier == "Bottom 50%":
        return "Critical"
    if m365_tier == "10-25%" and cop_tier == "25-50%":
        return "Medium"
    if m365_tier == "10-25%" and cop_tier == "Bottom 50%":
        return "High"
    if m365_tier in {"25-50%", "Bottom 50%"} and cop_tier in top2:
        return "Promoter"
    if m365_tier == "25-50%" and cop_tier == "25-50%":
        return "Medium"
    if m365_tier == "25-50%" and cop_tier == "Bottom 50%":
        return "Medium"
    return "Low"


def seg_fn(days: int, window_days: int) -> str:
    """Map active-day count to a per-week engagement-segment label.

    Normalizes to active days per week so labels mean the same thing regardless
    of pull length: <1, 1-2, 3-4, 5+ days/week. Window_days is the calendar
    span (max date - min date + 1) of the rolled-up data; 0 yields No Usage.
    """
    if window_days <= 0 or days <= 0:
        return "0. No Usage"
    rate = days * 7.0 / window_days
    if rate < 1.0:
        return "1. <1 Day/Week (Light)"
    if rate < 3.0:
        return "2. 1-2 Days/Week (Moderate)"
    if rate < 5.0:
        return "3. 3-4 Days/Week (Frequent)"
    return "4. 5+ Days/Week (Daily)"


def compute_ranks(values_by_uid: dict[str, float]) -> dict[str, int]:
    """
    0-based ascending rank via stable sort.  Ties get different sequential
    indices (matches PQ Table.Sort + Table.AddIndexColumn behaviour).
    """
    sorted_uids = sorted(values_by_uid.items(), key=lambda x: x[1])
    return {uid: i for i, (uid, _) in enumerate(sorted_uids)}


def _compute_copilot_event_count(
    ced: dict,
    operation: str,
    prompt_filter: str | None,
) -> int:
    """
    Compute the number of exploded rows a Copilot record would produce,
    using the same array-length logic as v1 (explode_copilot_record),
    but WITHOUT materializing any row dicts.
    Returns 0 if prompt_filter eliminates all messages (record skipped).
    """
    messages = get_array_fast(ced, "Messages")
    contexts = get_array_fast(ced, "Contexts")
    resources = get_array_fast(ced, "AccessedResources")
    plugins_raw = get_array_fast(ced, "AISystemPlugin")
    model_det_raw = get_array_fast(ced, "ModelTransparencyDetails")
    sensitivity_labels = get_array_fast(ced, "SensitivityLabels")

    # Prompt filtering (same logic as v1 lines 571-581)
    if prompt_filter:
        pf_lower = prompt_filter.lower()
        if pf_lower == "null":
            messages = [m for m in messages if safe_get(m, "isPrompt") is None]
        elif pf_lower == "both":
            messages = [m for m in messages if safe_get(m, "isPrompt") is not None]
        elif pf_lower == "prompt":
            messages = [m for m in messages if safe_get(m, "isPrompt") is True]
        elif pf_lower == "response":
            messages = [m for m in messages if safe_get(m, "isPrompt") is False]
        if not messages:
            return 0  # record filtered out entirely

    # Context items max for CopilotInteraction
    context_items_max: int = 0
    if operation == "CopilotInteraction" and contexts:
        for ctx in contexts:
            if ctx:
                items = get_array_fast(ctx, "Items")
                if items and len(items) > context_items_max:
                    context_items_max = len(items)

    if prompt_filter:
        row_count = max(1, len(messages))
    else:
        array_counts = [
            1, len(messages), len(contexts), len(resources),
            len(sensitivity_labels), len(plugins_raw), len(model_det_raw),
        ]
        if context_items_max > 0:
            array_counts.append(context_items_max)
        row_count = max(array_counts)

    return min(max(row_count, 1), EXPLOSION_PER_RECORD_ROW_CAP)


def _count_mail_items_accessed(audit_data: dict) -> int:
    """Items represented by one MailItemsAccessed event.
    Sums len(Folders[].FolderItems[]) when present; falls back to 1 (Bind-style)."""
    folders = audit_data.get("Folders")
    if isinstance(folders, list):
        total = 0
        for fld in folders:
            if not isinstance(fld, dict):
                continue
            fi = fld.get("FolderItems")
            if isinstance(fi, list):
                total += len(fi)
        if total > 0:
            return total
    return 1


_UPN_LOCAL_RE = re.compile(r"^[^\s\\@]+$")


_BARE_GUID_RE = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$", re.I)


def _is_human_upn(uid: str) -> bool:
    """True iff uid is a syntactically valid human UPN (local-part @ domain . tld), excluding
    well-known service/bot patterns (SupervisoryReview{...}@..., bare GUIDs)."""
    if not uid:
        return False
    s = uid.strip()
    if _BARE_GUID_RE.match(s):
        return False
    if s.lower().startswith("supervisoryreview{"):
        return False
    if "@" not in s or s.count("@") != 1:
        return False
    local, domain = s.split("@", 1)
    if not _UPN_LOCAL_RE.match(local):
        return False
    if "." not in domain or not domain or domain.startswith(".") or domain.endswith("."):
        return False
    return True


def _extract_rollup_keys(
    record: dict,
    audit_data: dict,
    ced: dict | None,
    prompt_filter: str | None = None,
) -> tuple[GroupKey, int, int, str, str, bool] | None:
    """
    Extract rollup group key + event count + items-accessed count + creation_time +
    original UserId + is_agent_interaction flag.

    Returns None if the record is filtered out (e.g. prompt_filter eliminates all messages).
    Returns:
        (group_key, event_count, items_accessed_count, creation_time_iso,
         original_user_id, is_agent_interaction)
    where group_key uses lowercased UserId for case-insensitive grouping and
    includes (agent_id, agent_name, context_type) so multi-agent users don't collapse.
    """
    # UserId: original casing preserved for output; lowered for grouping key
    raw_uid = _norm_key_str(safe_get(audit_data, "UserId") or record.get("UserId", ""))
    # Filter non-human/system identities (Teams Sync, ServicePrincipals, SIDs, bots, etc.)
    if not _is_human_upn(raw_uid):
        return None
    uid_lower = raw_uid.lower()

    # CreationDate: from CSV, normalized to midnight
    creation_date = normalize_creation_date(record.get("CreationDate", ""))

    # Operation: from audit_data → CSV fallback, preserve case, then canonicalize via OP_RENAME
    operation = _norm_key_str(
        safe_get(audit_data, "Operation") or record.get("Operation", "") or record.get("Operations", "")
    )
    operation = OP_RENAME.get(operation, operation)

    # Workload: from audit_data, preserve case
    workload = _norm_key_str(safe_get(audit_data, "Workload"))

    # SourceFileExtension: lowercased for DAX LOWER() compatibility
    sfe = _norm_key_str(safe_get(audit_data, "SourceFileExtension")).lower()

    # AppHost: from CED for Copilot, empty otherwise, preserve case
    if ced:
        app_host = _norm_key_str(
            safe_get(ced, "AppHost") or safe_get(audit_data, "AppHost")
        )
    else:
        app_host = _norm_key_str(safe_get(audit_data, "AppHost"))

    # Agent telemetry: from CopilotEventData when present, fall back to top-level AuditData fields
    agent_id = _norm_key_str(
        (safe_get(ced, "AgentId") if ced else None)
        or safe_get(audit_data, "AgentId")
    )
    agent_name = _norm_key_str(
        (safe_get(ced, "AgentName") if ced else None)
        or safe_get(audit_data, "AgentName")
    )
    context_type = ""
    if ced:
        contexts = get_array_fast(ced, "Contexts")
        if contexts:
            # First context wins for the key; multi-context records still collapse
            # cleanly because event_count already reflects context-array length.
            first_ctx = contexts[0]
            if isinstance(first_ctx, dict):
                context_type = _norm_key_str(safe_get(first_ctx, "Type"))

    # CreationTime: from audit_data, ISO formatted for lexicographic MIN/MAX
    creation_time = format_date_purview(safe_get(audit_data, "CreationTime"))

    # Event count
    if ced:
        event_count = _compute_copilot_event_count(ced, operation, prompt_filter)
        if event_count == 0:
            return None  # filtered out
    else:
        event_count = 1  # Non-Copilot: always 1:1

    # Items accessed count: only meaningful for MailItemsAccessed (Exchange).
    items_accessed_count = 0
    if operation == "MailItemsAccessed":
        items_accessed_count = _count_mail_items_accessed(audit_data)

    # IsAgentInteraction: TRUE iff AgentId present, AppHost is an agent surface,
    # or Operation is an agent op (AIAppInteraction).
    is_agent_interaction = bool(
        agent_id
        or app_host.lower() in AGENT_APPHOSTS
        or operation == "AIAppInteraction"
    )

    group_key: GroupKey = (
        uid_lower, creation_date, operation, workload, sfe, app_host,
        agent_id, agent_name, context_type,
    )
    return (
        group_key,
        event_count,
        items_accessed_count,
        creation_time,
        raw_uid,
        is_agent_interaction,
    )


def write_userstats_files(
    aggregated_csv_path: str | Path,
    userstats_csv_path: str | Path,
    session_csv_path: str | Path,
    quiet: bool,
    session_stats_csv_path: str | Path | None = None,
) -> tuple[int, int]:
    """
    Read the just-written aggregated rollup CSV and produce two additional files:
      *_UserStats.csv     — one row per unique UserId with pre-computed metrics
      *_SessionCohort.csv — one row per (UserId, AppColumn) with session cohort label

    When `session_stats_csv_path` is provided (v2.6.1+), the CECopilotPercentile_*
    columns are computed from per-user PromptCount (human interactions) instead of
    raw audit-event counts. This matches the AI in One semantics and prevents
    service-principal / plugin-chain inflation from skewing the CE Quadrant.

    Returns (user_count, session_cohort_row_count).
    """
    agg_path = Path(aggregated_csv_path)
    if not agg_path.is_file():
        if not quiet:
            print(f"[UserStats] WARNING: Aggregated CSV not found: {agg_path} — skipping.",
                  file=sys.stderr)
        return 0, 0

    userstats_path = Path(userstats_csv_path)
    session_path = Path(session_csv_path)

    t_start = time.perf_counter()

    # ── Per-user accumulators ────────────────────────────────────────────
    uid_original: dict[str, str] = {}          # uid_lower → first-seen casing
    cop_ec: dict[str, int] = defaultdict(int)
    m365_ec: dict[str, int] = defaultdict(int)
    ex_cop_ec: dict[str, int] = defaultdict(int)
    ex_m365_ec: dict[str, int] = defaultdict(int)

    t_days: dict[str, set[str]] = defaultdict(set)
    o_days: dict[str, set[str]] = defaultdict(set)
    w_days: dict[str, set[str]] = defaultdict(set)
    x_days: dict[str, set[str]] = defaultdict(set)
    p_days: dict[str, set[str]] = defaultdict(set)

    t_ec: dict[str, int] = defaultdict(int)
    o_ec: dict[str, int] = defaultdict(int)
    off_ec: dict[str, int] = defaultdict(int)

    # ── v2.5.0: DAX-aligned per-user raw activity counts, computed per window.
    # Three windows (L30, L60, Full) feed both the LP <App> Weighted measures and the
    # CE percentile ranks. Each is a dict keyed by window code → {uid: count}.
    def _wbuckets() -> dict[str, dict[str, int]]:
        return {w: defaultdict(int) for w in RANK_WINDOWS}
    teams_raw = _wbuckets()
    outlook_raw = _wbuckets()
    word_raw = _wbuckets()
    excel_raw = _wbuckets()
    ppt_raw = _wbuckets()
    copilot_chat_raw = _wbuckets()        # Operation = "CopilotInteraction" (LP)
    ce_copilot_raw = _wbuckets()          # broad: Workload="Copilot" OR Op contains "CopilotInteraction" (CE)

    session_ops: dict[tuple[str, str], set[str]] = defaultdict(set)

    # Track every distinct CreationDate seen in the rollup so we can
    # derive the data-window span (max - min + 1 calendar days) and
    # normalize the engagement segmentation to active-days-per-week.
    all_dates: set[str] = set()

    # ── v2.5.0: Pass 1 — determine the trailing-window cutoffs ──────────
    # Scan CreationDate only to find the most-recent date in the rollup. Cutoffs
    # are inclusive lower bounds; a row qualifies for window W iff date_key >= cutoff[W].
    # The "Full" window has no cutoff and always qualifies.
    _d_max_str = ""
    with open(agg_path, "r", encoding="utf-8-sig", newline="") as _f:
        _r = csv.DictReader(_f)
        for _row in _r:
            _dk = (_row.get("CreationDate", "") or "")[:10]
            if _dk and _dk > _d_max_str:
                _d_max_str = _dk
    if _d_max_str:
        try:
            _d_max = date.fromisoformat(_d_max_str)
            cutoff_l30 = (_d_max - timedelta(days=29)).isoformat()
            cutoff_l60 = (_d_max - timedelta(days=59)).isoformat()
        except ValueError:
            # Bad date — fall back to "everything qualifies"
            cutoff_l30 = ""
            cutoff_l60 = ""
    else:
        # No data — sentinel that nothing qualifies for L30/L60 (Full still does)
        cutoff_l30 = "9999-12-31"
        cutoff_l60 = "9999-12-31"
    if not quiet:
        print(f"[UserStats] Window cutoffs: L30 >= {cutoff_l30 or '(all)'}, "
              f"L60 >= {cutoff_l60 or '(all)'}, max date = {_d_max_str or '(none)'}")

    # ── Stream through aggregated CSV ────────────────────────────────────
    row_count = 0
    with open(agg_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            row_count += 1

            user_id = row.get("UserId", "")
            uid_lower = user_id.lower()
            if uid_lower not in uid_original:
                uid_original[uid_lower] = user_id

            date_key = row.get("CreationDate", "")[:10]   # YYYY-MM-DD
            op = row.get("Operation", "")
            wl = row.get("Workload", "")
            ext = (row.get("SourceFileExtension", "") or "").lower()
            app_host = (row.get("AppHost", "") or "").lower()

            if date_key:
                all_dates.add(date_key)

            try:
                event_count = int(row.get("EventCount", "1") or "1")
            except (ValueError, TypeError):
                event_count = 1

            copilot = is_copilot(op, wl)
            excel_file = is_excel_file_op(ext, op)

            # Core event counts
            if copilot:
                cop_ec[uid_lower] += event_count
            else:
                m365_ec[uid_lower] += event_count

            # ExCopEC: Copilot interactions in Excel (via AppHost); ExM365EC: Excel file ops by non-Copilot
            if copilot and app_host == "excel":
                ex_cop_ec[uid_lower] += 1        # row count, not EventCount
            if excel_file and not copilot:
                ex_m365_ec[uid_lower] += 1       # row count, not EventCount

            # Active days (distinct CreationDate values)
            if wl == "MicrosoftTeams" and op in TEAMS_OPS:
                t_days[uid_lower].add(date_key)
            if wl == "Exchange" and op in OUTLOOK_OPS:
                o_days[uid_lower].add(date_key)
            if ext in WORD_EXTS and op in FILE_OPS:
                w_days[uid_lower].add(date_key)
            if ext in EXCEL_EXTS and op in FILE_OPS:
                x_days[uid_lower].add(date_key)
            if ext in PPT_EXTS and op in FILE_OPS:
                p_days[uid_lower].add(date_key)

            # Activity event counts
            if wl == "MicrosoftTeams" and op in TEAMS_OPS:
                t_ec[uid_lower] += event_count
            if wl == "Exchange" and op in OUTLOOK_OPS:
                o_ec[uid_lower] += event_count
            if ext in OFFICE_EXTS and op in FILE_OPS:
                off_ec[uid_lower] += event_count

            # ── v2.5.0: DAX-aligned raw counts, accumulated per window.
            # Helper closure: write to Full always; to L60/L30 only if the row's
            # date_key satisfies the trailing-window cutoff.
            def _bump(buckets: dict[str, dict[str, int]], n: int) -> None:
                buckets["Full"][uid_lower] += n
                if date_key >= cutoff_l60:
                    buckets["L60"][uid_lower] += n
                if date_key >= cutoff_l30:
                    buckets["L30"][uid_lower] += n

            if wl == "MicrosoftTeams" and op in DAX_TEAMS_OPS:
                _bump(teams_raw, event_count)
            if wl == "Exchange" and op in DAX_OUTLOOK_OPS:
                _bump(outlook_raw, event_count)
            if op in DAX_FILE_OPS:
                if ext in WORD_EXTS:
                    _bump(word_raw, event_count)
                elif ext in EXCEL_EXTS:
                    _bump(excel_raw, event_count)
                elif ext in PPT_EXTS:
                    _bump(ppt_raw, event_count)
            if op == "CopilotInteraction":
                _bump(copilot_chat_raw, event_count)
            # CE Copilot Percentile filter: Workload="Copilot" OR Operation contains "CopilotInteraction"
            if wl == "Copilot" or "CopilotInteraction" in op:
                _bump(ce_copilot_raw, event_count)

            # Session cohort: distinct active dates per (user, app)
            app = app_column(ext, op, wl)
            if app != "M365 All Apps":
                session_ops[(uid_lower, app)].add(date_key)

    if row_count == 0:
        if not quiet:
            print("[UserStats] Aggregated CSV has 0 rows — skipping.")
        return 0, 0

    # ── Data-window span ────────────────────────────────────────────────
    # Calendar-day span between the earliest and latest CreationDate in the
    # rollup, inclusive. Used to normalize per-app engagement segments to
    # active-days-per-week, so labels mean the same thing whether the pull
    # covers 8 days, 30 days, or 6 months.
    if all_dates:
        try:
            d_min = min(all_dates)
            d_max = max(all_dates)
            window_days = (
                date.fromisoformat(d_max) - date.fromisoformat(d_min)
            ).days + 1
        except ValueError:
            window_days = max(len(all_dates), 1)
    else:
        window_days = 1
    if not quiet:
        print(f"[UserStats] Data window: {window_days} calendar day(s) "
              f"({d_min if all_dates else '?'} -> {d_max if all_dates else '?'})")

    # ── Percentile thresholds ────────────────────────────────────────────
    all_uids = sorted(uid_original.keys())
    total_users = len(all_uids)

    cop_vals = [cop_ec.get(u, 0) for u in all_uids]
    m365_vals = [m365_ec.get(u, 0) for u in all_uids]
    ex_cop_vals = [ex_cop_ec.get(u, 0) for u in all_uids]
    ex_m365_vals = [ex_m365_ec.get(u, 0) for u in all_uids]

    cop_p90 = percentile_inc(cop_vals, 0.90)
    cop_p75 = percentile_inc(cop_vals, 0.75)
    cop_p50 = percentile_inc(cop_vals, 0.50)

    m365_p90 = percentile_inc(m365_vals, 0.90)
    m365_p75 = percentile_inc(m365_vals, 0.75)
    m365_p50 = percentile_inc(m365_vals, 0.50)

    exc_p90 = percentile_inc(ex_cop_vals, 0.90)
    exc_p75 = percentile_inc(ex_cop_vals, 0.75)
    exc_p50 = percentile_inc(ex_cop_vals, 0.50)

    exm_p90 = percentile_inc(ex_m365_vals, 0.90)
    exm_p75 = percentile_inc(ex_m365_vals, 0.75)
    exm_p50 = percentile_inc(ex_m365_vals, 0.50)

    # ── Ranks ────────────────────────────────────────────────────────────
    # Copilot rank: computed only among Copilot users so the range maps to [0, 1]
    # within that group. Non-Copilot users are hardcoded to 0.0 downstream.
    copilot_uids = [u for u in all_uids if cop_ec.get(u, 0) > 0]
    copilot_user_count = len(copilot_uids)
    cop_rank = compute_ranks({u: cop_ec.get(u, 0) for u in copilot_uids})
    m365_rank = compute_ranks({u: m365_ec.get(u, 0) for u in all_uids})

    # ── v2.5.0: CE percentile ranks per window (integer 0–100, match DAX exactly) ──
    # DAX formula: ROUND( COUNTROWS(users with score <= mine) / COUNTROWS(users with score > 0) * 100 , 0)
    # Users with score 0 / no activity → BLANK (we emit empty string).
    def _ce_rank_pct(scores: dict[str, int]) -> dict[str, str]:
        """Return DAX-exact CE percentile rank per user, as a string ('' for BLANK)."""
        positives = sorted(v for v in scores.values() if v > 0)
        total = len(positives)
        out: dict[str, str] = {}
        if total == 0:
            return {u: "" for u in scores}
        for u, v in scores.items():
            if v <= 0:
                out[u] = ""
            else:
                below = bisect.bisect_right(positives, v)
                out[u] = str(round(below / total * 100))
        return out

    # M365 All Apps raw is derived per window (sum of 5 app raws). LP has no M365-AllApps
    # measure, so we only compute the rank — not stored as a column.
    m365_all_apps_raw = {
        w: {
            u: teams_raw[w].get(u, 0) + outlook_raw[w].get(u, 0) + word_raw[w].get(u, 0)
               + excel_raw[w].get(u, 0) + ppt_raw[w].get(u, 0)
            for u in all_uids
        }
        for w in RANK_WINDOWS
    }
    ce_rank_teams   = {w: _ce_rank_pct({u: teams_raw[w].get(u, 0)   for u in all_uids}) for w in RANK_WINDOWS}
    ce_rank_outlook = {w: _ce_rank_pct({u: outlook_raw[w].get(u, 0) for u in all_uids}) for w in RANK_WINDOWS}
    ce_rank_word    = {w: _ce_rank_pct({u: word_raw[w].get(u, 0)    for u in all_uids}) for w in RANK_WINDOWS}
    ce_rank_excel   = {w: _ce_rank_pct({u: excel_raw[w].get(u, 0)   for u in all_uids}) for w in RANK_WINDOWS}
    ce_rank_ppt     = {w: _ce_rank_pct({u: ppt_raw[w].get(u, 0)     for u in all_uids}) for w in RANK_WINDOWS}
    ce_rank_all     = {w: _ce_rank_pct(m365_all_apps_raw[w])                              for w in RANK_WINDOWS}

    # ── v2.6.1: CE Copilot Percentile based on PROMPT COUNT (human interactions) ──
    # Read the SessionStats CSV (if produced by run_rollup) and tally PromptCount per
    # user per window. This is the AI in One semantics: one count per `isPrompt=TRUE`
    # message — resistant to AI-response fanout, plugin chains, retries, and most
    # service-principal noise. Falls back to audit-event tally if SessionStats is
    # missing (older script invocations).
    prompt_raw = _wbuckets()
    if session_stats_csv_path:
        _ss = Path(session_stats_csv_path)
        if _ss.is_file():
            with open(_ss, "r", encoding="utf-8-sig", newline="") as _f:
                _r = csv.DictReader(_f)
                for _row in _r:
                    _uid = (_row.get("UserId") or "").strip().lower()
                    if not _uid:
                        continue
                    _date_key = (_row.get("CreationDate") or "")[:10]
                    try:
                        _pc = int(_row.get("PromptCount") or 0)
                    except ValueError:
                        _pc = 0
                    if _pc <= 0:
                        continue
                    prompt_raw["Full"][_uid] += _pc
                    if cutoff_l60 and _date_key >= cutoff_l60:
                        prompt_raw["L60"][_uid] += _pc
                    if cutoff_l30 and _date_key >= cutoff_l30:
                        prompt_raw["L30"][_uid] += _pc
            if not quiet:
                _tot = sum(prompt_raw["Full"].values())
                print(f"[UserStats] CE Copilot Percentile source: PromptCount "
                      f"({_tot:,} prompts across {len(prompt_raw['Full']):,} users)")
        else:
            if not quiet:
                print(f"[UserStats] WARNING: SessionStats CSV not found: {_ss} — "
                      f"falling back to audit-event count for CE Copilot Percentile.",
                      file=sys.stderr)
            prompt_raw = ce_copilot_raw  # fallback to legacy event-based percentile
    else:
        prompt_raw = ce_copilot_raw  # legacy mode (script invoked without SessionStats)

    ce_copilot_pct  = {w: _ce_rank_pct({u: prompt_raw[w].get(u, 0) for u in all_uids}) for w in RANK_WINDOWS}

    # ── Write *_UserStats.csv ────────────────────────────────────────────
    with open(userstats_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(USERSTATS_HEADER)

        for uid in all_uids:
            c_ec = cop_ec.get(uid, 0)
            m_ec = m365_ec.get(uid, 0)
            exc_ec = ex_cop_ec.get(uid, 0)
            exm_ec = ex_m365_ec.get(uid, 0)

            is_cop_user = "Copilot User" if c_ec > 0 else "Non-Copilot User"
            cop_tier = tier_fn(c_ec, cop_p90, cop_p75, cop_p50, zero_is_bottom=True)
            m365_tier = tier_fn(m_ec, m365_p90, m365_p75, m365_p50, zero_is_bottom=False)
            priority = priority_fn(m365_tier, cop_tier)

            ex_m365_tier = tier_fn(exm_ec, exm_p90, exm_p75, exm_p50, zero_is_bottom=False)
            ex_cop_tier = tier_fn(exc_ec, exc_p90, exc_p75, exc_p50, zero_is_bottom=False)
            excel_pri = priority_fn(ex_m365_tier, ex_cop_tier)

            cop_rank_val = 0.0 if c_ec == 0 else cop_rank[uid] / max(copilot_user_count, 1)
            m365_rank_val = m365_rank[uid] / total_users

            td = len(t_days.get(uid, set()))
            od = len(o_days.get(uid, set()))
            wd = len(w_days.get(uid, set()))
            xd = len(x_days.get(uid, set()))
            pd_ = len(p_days.get(uid, set()))

            t_act = t_ec.get(uid, 0)
            o_act = o_ec.get(uid, 0)
            off_act = off_ec.get(uid, 0)

            t_seg = "0. No Usage" if td == 0 else seg_fn(td, window_days)
            o_seg = "0. No Usage" if od == 0 else seg_fn(od, window_days)
            w_seg = "0. No Usage" if wd == 0 else seg_fn(wd, window_days)
            x_seg = "0. No Usage" if xd == 0 else seg_fn(xd, window_days)
            p_seg = "0. No Usage" if pd_ == 0 else seg_fn(pd_, window_days)

            office_days = wd + xd + pd_
            off_seg = "0. No Usage" if office_days == 0 else seg_fn(office_days, window_days)

            overall_days = len(
                t_days.get(uid, set()) | o_days.get(uid, set()) |
                w_days.get(uid, set()) | x_days.get(uid, set()) |
                p_days.get(uid, set())
            )
            overall_seg = "0. No Usage" if overall_days == 0 else seg_fn(overall_days, window_days)

            writer.writerow([
                uid_original[uid],
                c_ec, m_ec, exc_ec, exm_ec,
                is_cop_user, cop_tier, m365_tier,
                priority, excel_pri,
                f"{cop_rank_val:.6f}", f"{m365_rank_val:.6f}",
                td, od, wd, xd, pd_,
                t_act, o_act, off_act,
                t_seg, o_seg, w_seg, x_seg, p_seg,
                off_seg, overall_seg,
                # v2.5.0: precomputed raw + CE rank columns per window (order must
                # match USERSTATS_HEADER: 6 raws × 3 windows, then 7 ranks × 3 windows)
                teams_raw["L30"].get(uid, 0),   teams_raw["L60"].get(uid, 0),   teams_raw["Full"].get(uid, 0),
                outlook_raw["L30"].get(uid, 0), outlook_raw["L60"].get(uid, 0), outlook_raw["Full"].get(uid, 0),
                word_raw["L30"].get(uid, 0),    word_raw["L60"].get(uid, 0),    word_raw["Full"].get(uid, 0),
                excel_raw["L30"].get(uid, 0),   excel_raw["L60"].get(uid, 0),   excel_raw["Full"].get(uid, 0),
                ppt_raw["L30"].get(uid, 0),     ppt_raw["L60"].get(uid, 0),     ppt_raw["Full"].get(uid, 0),
                copilot_chat_raw["L30"].get(uid, 0), copilot_chat_raw["L60"].get(uid, 0), copilot_chat_raw["Full"].get(uid, 0),
                ce_rank_teams["L30"][uid],     ce_rank_teams["L60"][uid],     ce_rank_teams["Full"][uid],
                ce_rank_outlook["L30"][uid],   ce_rank_outlook["L60"][uid],   ce_rank_outlook["Full"][uid],
                ce_rank_word["L30"][uid],      ce_rank_word["L60"][uid],      ce_rank_word["Full"][uid],
                ce_rank_excel["L30"][uid],     ce_rank_excel["L60"][uid],     ce_rank_excel["Full"][uid],
                ce_rank_ppt["L30"][uid],       ce_rank_ppt["L60"][uid],       ce_rank_ppt["Full"][uid],
                ce_rank_all["L30"][uid],       ce_rank_all["L60"][uid],       ce_rank_all["Full"][uid],
                ce_copilot_pct["L30"][uid],    ce_copilot_pct["L60"][uid],    ce_copilot_pct["Full"][uid],
            ])

    # ── Write *_SessionCohort.csv ────────────────────────────────────────
    session_count = 0
    with open(session_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, lineterminator="\n")
        writer.writerow(SESSIONCOHORT_HEADER)

        for (uid, app), ops in sorted(session_ops.items()):
            n = len(ops)
            if n == 0:
                continue
            if n <= 5:
                cohort = "1-5 sessions"
            elif n <= 10:
                cohort = "6-10 sessions"
            elif n <= 20:
                cohort = "11-20 sessions"
            elif n <= 40:
                cohort = "21-40 sessions"
            elif n <= 60:
                cohort = "41-60 sessions"
            elif n <= 80:
                cohort = "61-80 sessions"
            else:
                cohort = "81+ sessions"
            writer.writerow([uid_original[uid], app, cohort])
            session_count += 1

    t_elapsed = time.perf_counter() - t_start

    if not quiet:
        print(f"[UserStats]     {total_users:,} users \u2192 {userstats_path.name} "
              f"({len(USERSTATS_HEADER)} columns)")
        print(f"[SessionCohort] {session_count:,} (user, app) pairs \u2192 {session_path.name}")
        print(f"[UserStats]     Elapsed: {t_elapsed:.2f}s")

    return total_users, session_count

## 5. Build User Stats and Session Cohort

The rollup and session tables are written out in the processor's CSV layout, then `write_userstats_files` runs on them unchanged.

In [ ]:
import csv, os, shutil, tempfile

WORK = tempfile.mkdtemp(prefix='m365_usage_')
paths = {name: os.path.join(WORK, f'{name}.csv')
         for name in ('Rollup', 'SessionStats', 'UserStats', 'SessionCohort', 'EntraUsers')}


def table_to_csv(table, header, path):
    """Stream a Delta table to a local CSV laid out exactly like the processor's own output."""
    n = 0
    with open(path, 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f, lineterminator='\n')
        w.writerow(header)
        if spark.catalog.tableExists(table):
            for row in spark.table(table).select(*header).toLocalIterator():
                w.writerow(['' if v is None else v for v in row])
                n += 1
    return n


n_roll = table_to_csv(ROLLUP_TABLE, ROLLUP_HEADER, paths['Rollup'])
n_sess = table_to_csv(SESSION_TABLE, SESSIONSTATS_HEADER, paths['SessionStats'])
print(f'Rollup rows: {n_roll:,}  Session stats rows: {n_sess:,}')
if n_roll == 0:
    raise RuntimeError(f'{ROLLUP_TABLE} is empty or missing; run M365_Usage_Audit_Ingester first.')

users_n, cohort_n = write_userstats_files(paths['Rollup'], paths['UserStats'], paths['SessionCohort'],
                                          quiet=False, session_stats_csv_path=paths['SessionStats'])
print(f'✓ UserStats: {users_n:,} users  SessionCohort: {cohort_n:,} rows')

# The template reads Entra Users as Windows-1252.
with open(paths['EntraUsers'], 'w', encoding='cp1252', errors='replace', newline='') as f:
    w = csv.writer(f, lineterminator='\n')
    w.writerow(ENTRA_COLUMNS)
    w.writerows(entra_rows)
print(f'✓ EntraUsers: {len(entra_rows):,} users')

## 6. Write CSVs and Delta tables

Point the template's five file parameters at the CSVs in `Files/m365_usage_export/` (see `08-report-templates`).

In [ ]:
import notebookutils

fs = notebookutils.fs
if not fs.exists(EXPORT_DIR):
    fs.mkdirs(EXPORT_DIR)
for name, local in paths.items():
    target = f'{EXPORT_DIR}/{name}.csv'
    if fs.exists(target):
        fs.rm(target)
    if fs.cp('file:' + os.path.abspath(local), target) is False:
        raise RuntimeError(f'Failed to copy {name}.csv to {target}.')
print(f'✓ Wrote {len(paths)} CSVs to {EXPORT_DIR}/')

for name, table, encoding in (('UserStats', USERSTATS_TABLE, 'utf-8'),
                              ('SessionCohort', COHORT_TABLE, 'utf-8'),
                              ('EntraUsers', ENTRA_TABLE, 'cp1252')):
    df = (spark.read.option('header', True).option('encoding', encoding)
          .option('multiLine', True).option('escape', '"').csv(f'{EXPORT_DIR}/{name}.csv'))
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(table)
    print(f'✓ {table}: {spark.table(table).count():,} rows')

shutil.rmtree(WORK, ignore_errors=True)